In [1]:
from google.colab import files
import pandas as pd
import io

print("Please upload your 'Crop_recommendation_2.csv' file:")
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nSuccessfully loaded {file_name}!")
df.head(2)

Please upload your 'Crop_recommendation_2.csv' file:


Saving Crop_recommendation.csv to Crop_recommendation.csv

Successfully loaded Crop_recommendation.csv!


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Separate features (X) and target (y)
X_raw = df.drop(columns=['label'])

# Convert the text crop names (e.g., 'rice', 'maize') into integer IDs for PyTorch
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(df['label'])

# 1. SPLIT FIRST
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_encoded, test_size=0.2, random_state=42)

# 2. SCALE SECOND (Fit ONLY on training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Transform only, do not fit

print(f"Data scaled safely without leakage. Training data shape: {X_train_scaled.shape}")

Data scaled safely without leakage. Training data shape: (1760, 7)


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

dev = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Define the Custom Dataset class
class FarmDS(Dataset):
    def __init__(self, X, y):
        # Convert numerical arrays to tensors and push them to the GPU
        self.X = torch.tensor(X, dtype=torch.float32, device=dev)
        self.y = torch.tensor(y, dtype=torch.long, device=dev)

    def __len__(self):
        # Tells PyTorch the total number of rows
        return len(self.X)

    def __getitem__(self, i):
        # Tells PyTorch how to fetch a single row
        return self.X[i], self.y[i]

# 2. Initialize the dataset with your training data
train_dataset = FarmDS(X_train_scaled, y_train)

# 3. Create the DataLoader (batches of 32, shuffled)
dl = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("DataLoader initialized! Fetching sample batches:\n")

# 4. Loop through the DataLoader to prove it works
for i, (xb, yb) in enumerate(dl):
    print(f"Batch {i:2d} | X feature shape: {list(xb.shape)} | y label shape: {list(yb.shape)}")

    # We will stop after printing the first 5 batches so it doesn't flood your screen
    if i == 4:
        break

print("\nData Pipeline is fully functional!")

DataLoader initialized! Fetching sample batches:

Batch  0 | X feature shape: [32, 7] | y label shape: [32]
Batch  1 | X feature shape: [32, 7] | y label shape: [32]
Batch  2 | X feature shape: [32, 7] | y label shape: [32]
Batch  3 | X feature shape: [32, 7] | y label shape: [32]
Batch  4 | X feature shape: [32, 7] | y label shape: [32]

Data Pipeline is fully functional!
